In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
import warnings

builder = (
    SparkSession.builder
    .appName("delta-minio-test")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        ",".join([
            "io.delta:delta-spark_2.12:3.1.0",
            "org.apache.hadoop:hadoop-aws:3.3.4",
            "com.amazonaws:aws-java-sdk-bundle:1.12.262",
        ])
    )
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minio")
    .config("spark.hadoop.fs.s3a.secret.key", "minio123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

spark

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-09728cbf-23fd-4f73-9d78-febc4eba4df1;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-spark_2.12/3.1.0/delta-spark_2.12-3.1.0.jar ...
	[SUCCESSFUL ] io.delta#delta-spark_2.12;3.1.0!delta-spark_2.12.jar (294ms)
downloading https://repo1.maven.org/maven2/io/delta/delta-storage/3.1.0/delta-storage-3.1.0.jar ...
	[SUCCESSFUL ] io.delta#delta-storage;3.1.0!delta-storage.jar (46ms)
downloading https://repo1.maven.org/maven2/org/antlr/antlr4-runtime/4.9.3/antlr4-runtime-4.9.3.jar ...
	[SUCCESSFUL ] org.antlr#antlr4-runtime;4.9.3!antlr4-runtime.jar (60ms)
:: resolution report :: resolve 1091ms :: artifacts dl 40

In [2]:
data = [
    ("XAUUSD", "2026-03-20 15:30:00", 3045.10),
    ("XAUUSD", "2026-03-20 15:31:00", 3045.25),
    ("XAUUSD", "2026-03-20 15:32:00", 3044.95),
]

df = spark.createDataFrame(data, ["symbol", "event_ts", "price_usd"])
df.show()

+------+-------------------+---------+
|symbol|           event_ts|price_usd|
+------+-------------------+---------+
|XAUUSD|2026-03-20 15:30:00|   3045.1|
|XAUUSD|2026-03-20 15:31:00|  3045.25|
|XAUUSD|2026-03-20 15:32:00|  3044.95|
+------+-------------------+---------+



In [3]:
df.write.mode("overwrite").parquet("s3a://lakehouse/bronze/gold_ticks_parquet_test/")

26/03/21 00:16:10 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [4]:
df2 = spark.read.parquet("s3a://lakehouse/bronze/gold_ticks_parquet_test/")
df2.show()

+------+-------------------+---------+
|symbol|           event_ts|price_usd|
+------+-------------------+---------+
|XAUUSD|2026-03-20 15:30:00|   3045.1|
|XAUUSD|2026-03-20 15:31:00|  3045.25|
|XAUUSD|2026-03-20 15:32:00|  3044.95|
+------+-------------------+---------+



In [5]:
spark = SparkSession.builder.master("local[*]").appName("version-check").getOrCreate()

print("Spark:", spark.version)
print("Hadoop:", spark._jvm.org.apache.hadoop.util.VersionInfo.getVersion())

Spark: 3.5.1
Hadoop: 3.3.4


26/03/21 00:16:18 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [6]:
delta_path = "s3a://lakehouse/bronze/gold_ticks_delta_test/"

df.write.format("delta").mode("overwrite").save(delta_path)

print("delta written to:", delta_path)

26/03/21 00:16:26 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


delta written to: s3a://lakehouse/bronze/gold_ticks_delta_test/


In [9]:
df_delta = spark.read.format("delta").load("s3a://lakehouse/bronze/gold_ticks_delta_test/")
df_delta.show()

+------+-------------------+---------+
|symbol|           event_ts|price_usd|
+------+-------------------+---------+
|XAUUSD|2026-03-20 15:30:00|   3045.1|
|XAUUSD|2026-03-20 15:31:00|  3045.25|
|XAUUSD|2026-03-20 15:32:00|  3044.95|
+------+-------------------+---------+



In [10]:
new_data = [
    ("XAUUSD", "2026-03-20 15:33:00", 3045.40),
    ("XAUUSD", "2026-03-20 15:34:00", 3045.55),
]

df_new = spark.createDataFrame(new_data, ["symbol", "event_ts", "price_usd"])

df_new.write.format("delta").mode("append").save("s3a://lakehouse/bronze/gold_ticks_delta_test/")

In [11]:
spark.read.format("delta").load("s3a://lakehouse/bronze/gold_ticks_delta_test/").show()

+------+-------------------+---------+
|symbol|           event_ts|price_usd|
+------+-------------------+---------+
|XAUUSD|2026-03-20 15:30:00|   3045.1|
|XAUUSD|2026-03-20 15:31:00|  3045.25|
|XAUUSD|2026-03-20 15:34:00|  3045.55|
|XAUUSD|2026-03-20 15:32:00|  3044.95|
|XAUUSD|2026-03-20 15:33:00|   3045.4|
+------+-------------------+---------+



In [12]:
import boto3
from botocore.client import Config

s3 = boto3.client(
    "s3",
    endpoint_url="http://minio:9000",
    aws_access_key_id="minio",
    aws_secret_access_key="minio123",
    config=Config(signature_version="s3v4"),
    region_name="us-east-1",
)

resp = s3.list_objects_v2(
    Bucket="lakehouse",
    Prefix="bronze/gold_ticks_delta_test/"
)

for obj in resp.get("Contents", []):
    print(obj["Key"])

bronze/gold_ticks_delta_test/_delta_log/00000000000000000000.json
bronze/gold_ticks_delta_test/_delta_log/00000000000000000001.json
bronze/gold_ticks_delta_test/part-00000-7f1df48d-ae78-4315-a3f4-e25c8d629eec-c000.snappy.parquet
bronze/gold_ticks_delta_test/part-00000-abf0f9b1-0798-48b8-a1c3-c772a65cf74f-c000.snappy.parquet
bronze/gold_ticks_delta_test/part-00007-b6176797-efa3-48be-b3c6-9f9e7ccd2418-c000.snappy.parquet
bronze/gold_ticks_delta_test/part-00011-112f58db-c886-43ca-88ef-0627346e24c2-c000.snappy.parquet
bronze/gold_ticks_delta_test/part-00015-45db2591-16d0-4b68-9ca8-35b090121ccc-c000.snappy.parquet
bronze/gold_ticks_delta_test/part-00023-3edf39e1-52c8-4279-a1f4-309c56c4d856-c000.snappy.parquet
bronze/gold_ticks_delta_test/part-00023-d6de3a0e-049e-41d0-a6d3-6a304793f3c3-c000.snappy.parquet


In [16]:
spark.sql("""
SELECT *
FROM delta.`s3a://lakehouse/bronze/gold_ticks_delta_test/`
""").show()

+------+-------------------+---------+
|symbol|           event_ts|price_usd|
+------+-------------------+---------+
|XAUUSD|2026-03-20 15:30:00|   3045.1|
|XAUUSD|2026-03-20 15:31:00|  3045.25|
|XAUUSD|2026-03-20 15:34:00|  3045.55|
|XAUUSD|2026-03-20 15:32:00|  3044.95|
|XAUUSD|2026-03-20 15:33:00|   3045.4|
+------+-------------------+---------+



In [18]:
spark.sql("""
CREATE TABLE gold_ticks_delta
USING DELTA
LOCATION 's3a://lakehouse/bronze/gold_ticks_delta_test/'
""")

DataFrame[]

In [19]:
spark.sql("""
DESCRIBE HISTORY gold_ticks_delta
""").show()

+-------+-------------------+------+--------+---------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|version|          timestamp|userId|userName|operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+-------------------+------+--------+---------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|      1|2026-03-21 00:19:05|  NULL|    NULL|    WRITE|{mode -> Append, ...|NULL|    NULL|     NULL|          0|  Serializable|         true|{numFiles -> 3, n...|        NULL|Apache-Spark/3.5....|
|      0|2026-03-21 00:16:25|  NULL|    NULL|    WRITE|{mode -> Overwrit...|NULL|    NULL|     NULL|       NULL|  Serializable|        false|{numFiles -> 4, n...|        NULL|Apache-Spark/3.5....|
+-------+------

In [22]:
spark.sql("""
SELECT *
FROM gold_ticks_delta VERSION AS OF 0
""").show()

+------+-------------------+---------+
|symbol|           event_ts|price_usd|
+------+-------------------+---------+
|XAUUSD|2026-03-20 15:30:00|   3045.1|
|XAUUSD|2026-03-20 15:31:00|  3045.25|
|XAUUSD|2026-03-20 15:32:00|  3044.95|
+------+-------------------+---------+

